# OpenFOAM volume fields to Farassat 1A

This notebook defines one or more closed permeable spheres or cylinders, interpolates the OpenFOAM volume fields `p` and `U` onto each surface, and compares their F1A predictions at one or more observer locations. No separate command-line step is required.

## Scope and requirements

- The folder must contain `constant/polyMesh` and time directories containing the volume fields `p` and `U`.
- FoamAcoustics reads those files with VTK. It does not invoke OpenFOAM, require `foamPostProcess`, or read/create `system/controlDict`.
- VTK and PyVista are installed with FoamAcoustics; no OpenFOAM installation is required in the notebook environment.
- The generated sphere or capped cylinder is fixed, closed, and must remain inside the fluid mesh at every sampled time.
- The current F1A implementation is the stationary permeable-surface formulation and assumes constant density and zero mean-flow convection during acoustic propagation. A rotating solid-blade pressure-only calculation needs a separate moving-surface implementation.
- The uploaded pressure field uses kinematic-pressure dimensions, so set `density_kg_m3` to the density used by the CFD case.
- With physical propagation timing, the record must extend beyond the nearest-face delay before any arrival is visible, and beyond the farthest-face delay plus the desired analysis window for a fully arrived spectrum. The short uploaded subset is only an interpolation preview; use the same source times for every surface comparison.

In [ ]:
import math
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

from foamacoustics import (
    CylinderSurface,
    SphereSurface,
    F1ASolver,
    interpolate_openfoam_fields,
    pressure_to_oaspl,
    pressure_to_spl_spectrum,
)
from foamacoustics.plotting import (
    plot_result_pressure_comparison,
    plot_surface_pressure_preview,
    plot_trace_spectrum_comparison,
)

## 1. Configure the case, surfaces, and observers

Supply `surfaces` as a list of unique `(name, surface)` pairs and `observer_coordinates_m` as an `(n, 3)` tensor. The example radius is just outside this case's rotating non-conformal interface; do not place a permeable surface directly on a mesh interface.

In [ ]:
case_dir = Path(r"../kwSST")
surfaces = [
    # (
    #     "cylinder_r01905",
    #     CylinderSurface(
    #         radius_m=0.1905,
    #         length_m=0.20,
    #         center_m=(0.0, 0.0, 0.0),
    #         axis=(0.0, 1.0, 0.0),
    #         axial_divisions=32,
    #         azimuthal_divisions=96,
    #         cap_radial_divisions=16,
    #     ),
    # ),
    (
    "sphere_r020",
        SphereSurface(
            radius_m=0.20,
            center_m=(0.0, 0.0, 0.0),
            polar_divisions=48,
            azimuthal_divisions=96,
        ),
    ),
        (
    "sphere_r020",
        SphereSurface(
            radius_m=0.20,
            center_m=(0.0, 0.0, 0.0),
            polar_divisions=48,
            azimuthal_divisions=96,
        ),
    ),
        (
    "sphere_r020",
        SphereSurface(
            radius_m=0.20,
            center_m=(0.0, 0.0, 0.0),
            polar_divisions=48,
            azimuthal_divisions=96,
        ),
    ),
        (
    "sphere_r020",
        SphereSurface(
            radius_m=0.20,
            center_m=(0.0, 0.0, 0.0),
            polar_divisions=48,
            azimuthal_divisions=96,
        ),
    ),
]

time_selection = "0.10:0.18"  # Use None to sample all available times.
pressure_units = "kinematic"
density_kg_m3 = 1.225
pressure_reference_pa = 0.0
region = None

rpm = 7000.0
blade_count = 2
sound_speed_m_s = 343.0
# Keep physical arrival times aligned when comparing surfaces. For a preview only,
# response_delay_s=1e-5 normalizes the first arrival onto a short record, but those
# signatures do not share a physical arrival-time reference across surfaces.
response_delay_s = None
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
numerical_dtype = torch.float32
observer_coordinates_m = torch.tensor(
    [[1.88, 0.0, 0.0]],
    device=device,
    dtype=numerical_dtype,
)
if observer_coordinates_m.ndim != 2 or observer_coordinates_m.shape[1] != 3:
    raise ValueError("observer_coordinates_m must be an (n, 3) tensor.")
surface_names = [name for name, _ in surfaces]
if not surfaces or len(surface_names) != len(set(surface_names)):
    raise ValueError("surfaces must contain at least one uniquely named surface.")
vtk_read_workers = 2
chunk_size = 64
invalid_time_policy = "interpolate"
print(f"Compute device: {device}")

## 2. Check the case and generated surfaces

This checks the mesh layout needed by VTK and reports the resolution and bounds of the analytic surface. No OpenFOAM runtime files are needed.

In [ ]:
mesh_dir = case_dir / "constant"
if region is not None:
    mesh_dir /= region
mesh_dir /= "polyMesh"
required_files = [
    mesh_dir / name for name in ("points", "faces", "owner", "neighbour", "boundary")
]
missing_files = [
    path
    for path in required_files
    if not path.is_file() and not path.with_name(path.name + ".gz").is_file()
]
if missing_files:
    missing = "\n".join(f"  - {path}" for path in missing_files)
    raise FileNotFoundError(f"Incomplete OpenFOAM base mesh:\n{missing}")

print(f"Case: {case_dir.resolve()}")
surface_meshes = {}
for surface_name, surface in surfaces:
    surface_mesh = surface.triangulate()
    surface_meshes[surface_name] = surface_mesh
    bounds_min = surface_mesh.vertices_m.amin(dim=0)
    bounds_max = surface_mesh.vertices_m.amax(dim=0)
    print(f"\nSurface: {surface_name}")
    print(
        f"Vertices: {len(surface_mesh.vertices_m):,}; panels: {len(surface_mesh.triangles):,}"
    )
    print(f"Bounds [m]: {bounds_min.tolist()} to {bounds_max.tolist()}")
    print(f"Closed volume: {surface_mesh.signed_volume_m3:.9g} m^3")

## 3. Interpolate `p` and `U` onto the surface

The function reads each selected volume-field time directly with VTK. It uses VTK for cell location, then reconstructs OpenFOAM-style face centres, face-pyramid cell-volume centroids, inverse-distance cell-to-vertex values, and cell-centre/face-vertex tetrahedral weights. Candidate tetrahedra use a cell-volume-relative tolerance and Euclidean nearest-tetrahedron fallback. This is numerically close to OpenFOAM `cellPoint` in ordinary interior cells, but it is not bitwise identical because VTK controls cell location and face topology, and physical-boundary/non-conformal point constraints are not reconstructed. Results are stored in a source-signature-addressed VTK cache and returned as an `OpenFOAMSurfaceSeries`; the remaining F1A workflow is unchanged.

In [ ]:
interpolation_start = perf_counter()
series_by_surface = {}
for surface_name, surface in surfaces:
    surface_start = perf_counter()
    series = interpolate_openfoam_fields(
        case_dir,
        surface,
        surface_name=surface_name,
        time_selection=time_selection,
        pressure_units=pressure_units,
        density_kg_m3=density_kg_m3,
        pressure_reference_pa=pressure_reference_pa,
        region=region,
        dtype=numerical_dtype,
        assume_stationary_mesh=True,
        workers=vtk_read_workers,
    )
    series_by_surface[surface_name] = series
    print(
        f"\n{surface_name}: interpolation/cache validation {perf_counter() - surface_start:.2f} s"
    )
    print(f"Sampled VTK root: {series.input_dir}")
    print(f"Frames: {len(series):,}; panels: {series.panel_count:,}")
    print(f"Times: {series.time_names[0]} to {series.time_names[-1]} s")
print(f"Interpolation and validation: {perf_counter() - interpolation_start:.2f} s")
source_time_grids = {
    tuple(float(t) for t in series.absolute_times_s.tolist())
    for series in series_by_surface.values()
}
if len(source_time_grids) != 1:
    raise ValueError("Compared surfaces must use identical absolute source times.")
source_duration_s = min(
    float(series.source_times_s[-1]) for series in series_by_surface.values()
)
if source_duration_s < 60.0 / rpm:
    print(
        "NOTE: this source record is shorter than one shaft revolution; the interpolation preview is valid, but a spectrum needs more volume-field times."
    )

## 4. Build the binary cache and inspect the sampled fields

Each configured surface is checked against its source-signature-addressed sampled-VTK cache and its binary frame cache. Observer coordinates are deliberately absent from both cache keys: a new observer reuses an unchanged surface cache, then F1A propagation is recomputed for that observer.

In [ ]:
first_frames = {}
for surface_name, series in series_by_surface.items():
    cache_was_present = series.cache_enabled
    if not cache_was_present:
        cache_bar = tqdm(
            total=len(series), desc=f"Caching {surface_name}", unit="frame"
        )

        def report_cache_progress(done, total, message):
            cache_bar.update(done - cache_bar.n)

        try:
            series.build_cache(
                workers=vtk_read_workers,
                invalid_time_policy=invalid_time_policy,
                progress_callback=report_cache_progress,
            )
        finally:
            cache_bar.close()
    first_frame = next(series.iter_frames())
    first_frames[surface_name] = first_frame
    time_steps_s = torch.diff(series.absolute_times_s)
    area_closure_ratio = (
        torch.linalg.vector_norm(series.outward_area_vectors_m2.sum(dim=0))
        / series.face_areas_m2.sum()
    )
    cache_status = "hit" if cache_was_present else "built"
    print(f"\n{surface_name}: binary cache {cache_status}; {series.cache_dir}")
    print(f"Time-step range: {time_steps_s.min():.9g} to {time_steps_s.max():.9g} s")
    print(
        f"Total area: {series.face_areas_m2.sum():.9g} m^2; closure ratio: {area_closure_ratio:.3e}"
    )
    print(
        f"First-frame pressure: {first_frame.pressure_pa.min():.6g} to {first_frame.pressure_pa.max():.6g} Pa"
    )

### Preview the interpolated pressure

Check that the sampled surface is in the intended location and that its first pressure field is spatially sensible before running the more expensive acoustic solve.

In [ ]:
for surface_name, series in series_by_surface.items():
    plot_surface_pressure_preview(
        series.face_centers_m,
        first_frames[surface_name].pressure_pa,
        title=f"Sampled pressure: {surface_name}",
    )
    plt.show()

## 5. Run Farassat 1A

From here onward, the workflow is the same as for a previously sampled OpenFOAM VTK series.

In [ ]:
solve_start = perf_counter()
results_by_surface = {}
observer_for_delay_m = observer_coordinates_m.detach().cpu().to(torch.float64)
for surface_name, series in series_by_surface.items():
    physical_delays_s = (
        torch.linalg.vector_norm(
            series.face_centers_m.detach().cpu().to(torch.float64)[None, :, :]
            - observer_for_delay_m[:, None, :],
            dim=2,
        )
        / sound_speed_m_s
    )
    minimum_physical_delay_s = float(physical_delays_s.min())
    maximum_physical_delay_s = float(physical_delays_s.max())
    if response_delay_s is None:
        print(
            f"{surface_name}: physical delays {minimum_physical_delay_s * 1e3:.3f} to {maximum_physical_delay_s * 1e3:.3f} ms"
        )
        if source_duration_s <= minimum_physical_delay_s:
            raise RuntimeError(
                f"{surface_name}: source record ends before the first physical arrival."
            )
        if source_duration_s <= maximum_physical_delay_s:
            print(
                f"NOTE: {surface_name} has no interval in which every panel has arrived."
            )

    solver = F1ASolver(
        series,
        rpm=rpm,
        density_kg_m3=density_kg_m3,
        sound_speed_m_s=sound_speed_m_s,
        response_delay_s=response_delay_s,
        adjust_time_step=True,
        device=device,
        chunk_size=chunk_size,
    )
    solve_bar = tqdm(total=len(series), desc=f"F1A {surface_name}", unit="frame")

    def report_solver_progress(done, total, message):
        solve_bar.update(done - solve_bar.n)

    try:
        result = solver.predict(
            observer_coordinates_m,
            workers=vtk_read_workers,
            invalid_time_policy=invalid_time_policy,
            progress_callback=report_solver_progress,
            remove_mean=False,
            algorithm="auto",
        )
    finally:
        solve_bar.close()
    results_by_surface[surface_name] = result
    maximum_delay_ms = (
        torch.as_tensor(result.maximum_delay_s).detach().cpu().max() * 1e3
    )
    print(
        f"{surface_name}: {result.propagation_algorithm}; max delay {maximum_delay_ms:.6f} ms"
    )

print(f"Total F1A wall time: {perf_counter() - solve_start:.2f} s")
print(
    "Surface caches were reused where present; F1A observer propagation was computed above."
)

In [ ]:
observer_count = observer_coordinates_m.shape[0]
observer_titles = [
    f"Observer {index}: {coordinate.detach().cpu().tolist()} m"
    for index, coordinate in enumerate(observer_coordinates_m)
]
plot_result_pressure_comparison(
    results_by_surface,
    observer_labels=observer_titles,
    ylabel="Total pressure [Pa]",
)
plt.show()

## 6. Compute the fully arrived acoustic spectrum

Only the part of the observer trace after the maximum propagation delay is used. For quantitative comparisons, use the same fully arrived duration for every surface and preferably an integer number of shaft rotations.

In [ ]:
def linear_resample(source_times_s, source_values, query_times_s):
    """Linearly resample one pressure trace without leaving Torch."""
    query_times_s = query_times_s.to(
        device=source_times_s.device, dtype=source_times_s.dtype
    )
    if query_times_s[0] < source_times_s[0] or query_times_s[-1] > source_times_s[-1]:
        raise ValueError("The requested FFT grid extends beyond the observer record.")
    right = torch.searchsorted(source_times_s, query_times_s, right=True)
    right = right.clamp(1, source_times_s.numel() - 1)
    left = right - 1
    fraction = (query_times_s - source_times_s[left]) / (
        source_times_s[right] - source_times_s[left]
    )
    return source_values[left] + fraction * (source_values[right] - source_values[left])


shaft_period_s = 60.0 / rpm
shaft_frequency_hz = rpm / 60.0
blade_passing_frequency_hz = blade_count * shaft_frequency_hz
spectra_by_surface_observer = {}
for surface_name, result in results_by_surface.items():
    maximum_delays_s = (
        torch.as_tensor(result.maximum_delay_s).detach().cpu().reshape(-1)
    )
    for observer_index in range(observer_count):
        selected = result.for_observer(observer_index)
        spectrum_start_s = max(
            float(maximum_delays_s[observer_index]),
            float(selected.observer_times_s[0].detach().cpu()),
        )
        spectrum_end_s = float(selected.observer_times_s[-1].detach().cpu())
        available_rotations = math.floor(
            (spectrum_end_s - spectrum_start_s) / shaft_period_s + 1.0e-12
        )
        if available_rotations < 1:
            raise RuntimeError(
                f"{surface_name}, observer {observer_index}: less than one fully arrived shaft rotation."
            )
        native_dt_s = float(
            torch.median(torch.diff(selected.observer_times_s)).detach().cpu()
        )
        samples_per_rotation = max(2, round(shaft_period_s / native_dt_s))
        spectrum_dt_s = shaft_period_s / samples_per_rotation
        sample_count = available_rotations * samples_per_rotation
        spectrum_times_s = spectrum_start_s + spectrum_dt_s * torch.arange(
            sample_count, dtype=torch.float64
        )
        spectrum_pressure_pa = linear_resample(
            selected.observer_times_s, selected.total_pressure_pa, spectrum_times_s
        )
        spectrum = pressure_to_spl_spectrum(
            spectrum_pressure_pa, sampling_rate_hz=1.0 / spectrum_dt_s
        )
        spectra_by_surface_observer[(surface_name, observer_index)] = spectrum
        oaspl_db = float(pressure_to_oaspl(spectrum_pressure_pa).detach().cpu())
        print(
            f"{surface_name}, observer {observer_index}: {available_rotations} rotations, {oaspl_db:.3f} dB OASPL"
        )

## 7. Compare every surface and observer

Every surface/observer pressure history is resampled onto the intersection of the fully arrived time ranges, using a shared integer-revolution FFT record. Surface sampling caches are independent of observer coordinates; changing observers reuses those caches but reruns F1A propagation.

In [ ]:
surface_comparison = {}
for surface_name, result in results_by_surface.items():
    for observer_index in range(observer_count):
        coordinate = observer_coordinates_m[observer_index].detach().cpu().tolist()
        coordinate_label = "(" + ", ".join(f"{value:.2f}" for value in coordinate) + ")"
        label = f"{surface_name} | observer {observer_index} {coordinate_label} m"
        surface_comparison[label] = {
            "result": result.for_observer(observer_index),
            "panel_count": series_by_surface[surface_name].panel_count,
        }

common_start_s = max(
    max(
        float(torch.as_tensor(values["result"].maximum_delay_s).detach().cpu()),
        float(values["result"].observer_times_s[0].detach().cpu()),
    )
    for values in surface_comparison.values()
)
common_end_s = min(
    float(values["result"].observer_times_s[-1].detach().cpu())
    for values in surface_comparison.values()
)
coarsest_native_dt_s = max(
    float(torch.median(torch.diff(values["result"].observer_times_s)).detach().cpu())
    for values in surface_comparison.values()
)
common_rotations = math.floor(
    (common_end_s - common_start_s) / shaft_period_s + 1.0e-12
)
if common_rotations < 1:
    raise RuntimeError(
        "The common fully arrived record contains less than one shaft rotation."
    )

common_samples_per_rotation = max(2, round(shaft_period_s / coarsest_native_dt_s))
common_dt_s = shaft_period_s / common_samples_per_rotation
common_sample_count = common_rotations * common_samples_per_rotation
common_times_s = common_start_s + common_dt_s * torch.arange(
    common_sample_count, dtype=torch.float64
)
if 0.5 / common_dt_s <= blade_passing_frequency_hz:
    raise RuntimeError(
        "The common sampling rate does not resolve the blade-passing frequency."
    )

comparison_traces = {}
comparison_spectra = {}
comparison_metrics = {}
for name, values in surface_comparison.items():
    stored_result = values["result"]
    trace = linear_resample(
        stored_result.observer_times_s, stored_result.total_pressure_pa, common_times_s
    )
    stored_spectrum = pressure_to_spl_spectrum(
        trace, sampling_rate_hz=1.0 / common_dt_s
    )
    bpf_bin = int(
        torch.argmin(
            torch.abs(stored_spectrum.frequencies_hz - blade_passing_frequency_hz)
        )
    )
    centred_trace = trace - trace.mean()
    comparison_traces[name] = trace
    comparison_spectra[name] = stored_spectrum
    comparison_metrics[name] = {
        "rms_pa": float(torch.sqrt(torch.mean(centred_trace**2)).detach().cpu()),
        "oaspl_db": float(pressure_to_oaspl(trace).detach().cpu()),
        "bpf_spl_db": float(stored_spectrum.spl_db[bpf_bin].detach().cpu()),
    }

plot_trace_spectrum_comparison(
    common_times_s - common_start_s,
    comparison_traces,
    comparison_spectra,
    oaspl_db={
        name: metrics["oaspl_db"] for name, metrics in comparison_metrics.items()
    },
    shaft_frequency_hz=shaft_frequency_hz,
    blade_passing_frequency_hz=blade_passing_frequency_hz,
)
plt.show()

print(
    f"Common record: {common_rotations} rotations, {common_sample_count:,} samples, "
    f"{common_sample_count * common_dt_s:.6f} s"
)
print(
    f"{'surface':<24} {'panels':>10} {'RMS [Pa]':>12} {'OASPL [dB]':>12} {'BPF SPL [dB]':>14}"
)
for name, values in surface_comparison.items():
    metrics = comparison_metrics[name]
    print(
        f"{name:<24} {values['panel_count']:>10,} {metrics['rms_pa']:>12.5g} "
        f"{metrics['oaspl_db']:>12.3f} {metrics['bpf_spl_db']:>14.3f}"
    )